In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import pandas as pd

file_path = "/lakehouse/default/Files/bronze/Online Retail.xlsx"

raw_pd = pd.read_excel(file_path)

print("Rows:", len(raw_pd))
print("Columns:", raw_pd.columns.tolist())

display(raw_pd.head(10))

StatementMeta(, 2d8a40af-23ca-4ab9-975a-b269b778e9a2, 3, Finished, Available, Finished, False)

Rows: 541909
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


SynapseWidget(Synapse.DataFrame, 655ab8c1-6e8d-46e9-b5b6-98145c0e59ce)

In [2]:
# Create a Bronze area, then save the original data unchanged

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

bronze_df = spark.createDataFrame(raw_pd)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.online_retail_raw")

print("Bronze table created successfully.")

display(spark.sql("""
    SELECT *
    FROM bronze.online_retail_raw
    LIMIT 10
"""))

StatementMeta(, 2d8a40af-23ca-4ab9-975a-b269b778e9a2, 5, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Could not convert 'C536379' with type str: tried to convert to int64
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


Bronze table created successfully.


SynapseWidget(Synapse.DataFrame, 82277b6f-93f3-49aa-8452-33ac8f6c9c05)

In [3]:
from pyspark.sql import functions as F

bronze = spark.table("bronze.online_retail_raw")

total_rows = bronze.count()
duplicate_rows = total_rows - bronze.dropDuplicates().count()

quality_summary = bronze.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("CustomerID").isNull(), 1).otherwise(0)).alias("blank_customer_id"),
    F.sum(F.when(F.col("Description").isNull(), 1).otherwise(0)).alias("blank_description"),
    F.sum(F.when(F.col("InvoiceDate").isNull(), 1).otherwise(0)).alias("blank_invoice_date"),
    F.sum(F.when(F.col("Quantity") < 0, 1).otherwise(0)).alias("negative_quantity_rows"),
    F.sum(F.when(F.col("UnitPrice") < 0, 1).otherwise(0)).alias("negative_price_rows")
)

print("Duplicate full rows:", duplicate_rows)
display(quality_summary)

StatementMeta(, 2d8a40af-23ca-4ab9-975a-b269b778e9a2, 6, Finished, Available, Finished, False)

Duplicate full rows: 5268


SynapseWidget(Synapse.DataFrame, c8ca3913-7823-43e8-878d-76a966056b12)

In [4]:
from pyspark.sql import functions as F

# Create the Silver table group
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_df = (
    bronze
    .dropDuplicates()
    .withColumn("InvoiceNo", F.trim(F.col("InvoiceNo").cast("string")))
    .withColumn("StockCode", F.trim(F.col("StockCode").cast("string")))
    .withColumn("Description", F.coalesce(F.trim(F.col("Description")), F.lit("Unknown Product")))
    .withColumn("CustomerID", F.coalesce(F.col("CustomerID").cast("string"), F.lit("UNKNOWN")))
    .withColumn("InvoiceDate", F.to_timestamp(F.col("InvoiceDate")))
    .withColumn("Quantity", F.col("Quantity").cast("integer"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))
    .withColumn(
        "TransactionType",
        F.when(
            F.col("InvoiceNo").startswith("C") | (F.col("Quantity") < 0),
            F.lit("Return")
        ).otherwise(F.lit("Sale"))
    )
    # Remove only records that cannot be trusted for analysis
    .filter(F.col("InvoiceNo").isNotNull() & (F.col("InvoiceNo") != ""))
    .filter(F.col("StockCode").isNotNull() & (F.col("StockCode") != ""))
    .filter(F.col("InvoiceDate").isNotNull())
    .filter(F.col("Quantity") != 0)
    .filter(F.col("UnitPrice") >= 0)
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.online_retail_clean")

print("Silver table created.")
display(spark.sql("""
    SELECT
        TransactionType,
        COUNT(*) AS row_count,
        MIN(InvoiceDate) AS first_date,
        MAX(InvoiceDate) AS last_date
    FROM silver.online_retail_clean
    GROUP BY TransactionType
"""))

StatementMeta(, 2d8a40af-23ca-4ab9-975a-b269b778e9a2, 7, Finished, Available, Finished, False)

Silver table created.


SynapseWidget(Synapse.DataFrame, 25622185-6414-4aa4-a072-7002f04d5867)

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

clean = spark.table("silver.online_retail_clean")

# -----------------------------
# 1. DimDate
# -----------------------------
dim_date = (
    clean
    .select(F.to_date("InvoiceDate").alias("Date"))
    .where(F.col("Date").isNotNull())
    .dropDuplicates()
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
    .withColumn("DayName", F.date_format("Date", "EEEE"))
    .select(
        "DateKey", "Date", "Year", "Quarter",
        "MonthNumber", "MonthName", "DayOfMonth", "DayName"
    )
)

# -----------------------------
# 2. DimProduct
# Keep the latest known product description and price per StockCode
# -----------------------------
product_window = Window.partitionBy("StockCode").orderBy(F.col("InvoiceDate").desc())

dim_product = (
    clean
    .filter(F.col("StockCode").isNotNull())
    .withColumn("row_num", F.row_number().over(product_window))
    .filter(F.col("row_num") == 1)
    .select(
        F.col("StockCode").alias("ProductKey"),
        F.col("StockCode"),
        F.col("Description").alias("ProductName"),
        F.col("UnitPrice").alias("LatestUnitPrice")
    )
)

# -----------------------------
# 3. DimCustomer
# UNKNOWN is kept so sales with missing customer IDs are not lost
# -----------------------------
dim_customer = (
    clean
    .select(
        F.col("CustomerID").alias("CustomerKey"),
        F.col("CustomerID"),
        F.col("Country")
    )
    .dropDuplicates()
)

# -----------------------------
# 4. FactSales
# One row = one invoice line
# Negative values represent returns
# -----------------------------
fact_sales = (
    clean
    .withColumn("DateKey", F.date_format("InvoiceDate", "yyyyMMdd").cast("int"))
    .withColumn("ProductKey", F.col("StockCode"))
    .withColumn("CustomerKey", F.col("CustomerID"))
    .withColumn("SalesAmount", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .select(
        "InvoiceNo",
        "DateKey",
        "ProductKey",
        "CustomerKey",
        "Quantity",
        "UnitPrice",
        "SalesAmount",
        "TransactionType"
    )
)

# -----------------------------
# 5. FactInventory — simulated learning dataset
# This estimates stock using product demand, not real warehouse stock.
# -----------------------------
fact_inventory = (
    fact_sales
    .groupBy("ProductKey")
    .agg(
        F.sum(F.when(F.col("Quantity") > 0, F.col("Quantity")).otherwise(0)).alias("UnitsSold"),
        F.max("UnitPrice").alias("EstimatedUnitPrice")
    )
    .withColumn(
        "EstimatedStockOnHand",
        F.greatest(
            F.lit(0),
            F.round(F.col("UnitsSold") * F.lit(0.08)).cast("int")
        )
    )
    .withColumn(
        "InventoryValue",
        F.round(F.col("EstimatedStockOnHand") * F.col("EstimatedUnitPrice"), 2)
    )
    .withColumn(
        "InventoryStatus",
        F.when(F.col("EstimatedStockOnHand") <= 10, "Low Stock")
         .when(F.col("EstimatedStockOnHand") <= 30, "Medium Stock")
         .otherwise("Healthy Stock")
    )
    .withColumn("SnapshotDate", F.current_date())
    .select(
        "ProductKey",
        "SnapshotDate",
        "EstimatedStockOnHand",
        "EstimatedUnitPrice",
        "InventoryValue",
        "InventoryStatus"
    )
)

# Save all Gold tables
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.DimDate")
dim_product.write.format("delta").mode("overwrite").saveAsTable("gold.DimProduct")
dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold.DimCustomer")
fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold.FactSales")
fact_inventory.write.format("delta").mode("overwrite").saveAsTable("gold.FactInventory")

print("Gold star schema created successfully.")

display(spark.sql("""
    SELECT 'DimDate' AS table_name, COUNT(*) AS rows FROM gold.DimDate
    UNION ALL
    SELECT 'DimProduct', COUNT(*) FROM gold.DimProduct
    UNION ALL
    SELECT 'DimCustomer', COUNT(*) FROM gold.DimCustomer
    UNION ALL
    SELECT 'FactSales', COUNT(*) FROM gold.FactSales
    UNION ALL
    SELECT 'FactInventory', COUNT(*) FROM gold.FactInventory
"""))

StatementMeta(, 2d8a40af-23ca-4ab9-975a-b269b778e9a2, 8, Finished, Available, Finished, False)

Gold star schema created successfully.


SynapseWidget(Synapse.DataFrame, 8acc25bc-a4c8-4842-a426-dd770dca8f27)

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

clean = spark.table("silver.online_retail_clean")

product_window = Window.partitionBy("StockCode").orderBy(F.col("InvoiceDate").desc())

dim_product_updated = (
    clean
    .filter(F.col("StockCode").isNotNull())
    .withColumn("row_num", F.row_number().over(product_window))
    .filter(F.col("row_num") == 1)
    .withColumn("DescriptionUpper", F.upper(F.col("Description")))
    .withColumn(
        "ProductCategory",
        F.when(
            F.col("DescriptionUpper").contains("MUG") |
            F.col("DescriptionUpper").contains("BOWL") |
            F.col("DescriptionUpper").contains("PLATE") |
            F.col("DescriptionUpper").contains("SPOON") |
            F.col("DescriptionUpper").contains("TEA"),
            "Kitchen & Dining"
        )
        .when(
            F.col("DescriptionUpper").contains("CANDLE") |
            F.col("DescriptionUpper").contains("LAMP") |
            F.col("DescriptionUpper").contains("LIGHT") |
            F.col("DescriptionUpper").contains("LANTERN"),
            "Lighting & Candles"
        )
        .when(
            F.col("DescriptionUpper").contains("CHRISTMAS") |
            F.col("DescriptionUpper").contains("EASTER") |
            F.col("DescriptionUpper").contains("PARTY") |
            F.col("DescriptionUpper").contains("BIRTHDAY"),
            "Party & Seasonal"
        )
        .when(
            F.col("DescriptionUpper").contains("TOY") |
            F.col("DescriptionUpper").contains("DOLL") |
            F.col("DescriptionUpper").contains("GAME") |
            F.col("DescriptionUpper").contains("CHILD"),
            "Toys & Children"
        )
        .when(
            F.col("DescriptionUpper").contains("CARD") |
            F.col("DescriptionUpper").contains("PAPER") |
            F.col("DescriptionUpper").contains("PEN") |
            F.col("DescriptionUpper").contains("NOTEBOOK"),
            "Stationery & Office"
        )
        .when(
            F.col("DescriptionUpper").contains("BOX") |
            F.col("DescriptionUpper").contains("BASKET") |
            F.col("DescriptionUpper").contains("BAG") |
            F.col("DescriptionUpper").contains("TIN"),
            "Storage & Household"
        )
        .otherwise("Other")
    )
    .select(
        F.col("StockCode").alias("ProductKey"),
        "StockCode",
        F.col("Description").alias("ProductName"),
        "ProductCategory",
        F.col("UnitPrice").alias("LatestUnitPrice")
    )
)

dim_product_updated.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.DimProduct")

display(
    dim_product_updated.groupBy("ProductCategory")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, 9fb8fbf9-c542-4082-82fb-989fe4b8373d, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a9a3643-2748-472d-8b50-4b80b8d9a789)